# Epi Info AI RECORDLINK validation lab — V0.7

Independently reconstruct deterministic blocking, exact comparison, Jaro-Winkler similarities, six-point candidate scores, threshold classes, fingerprint-bound review evidence, and conflict-aware person clusters from the synthetic RECORDLINK project. Passing validates this bounded fixture; it does not approve automatic linkage or a production patient-matching method.

In [ ]:
import csv, io, json, re, unicodedata
from collections import Counter
from pyodide.http import pyfetch
async def get_text(path):
    response = await pyfetch(path); response.raise_for_status(); return await response.string()
source_a = list(csv.DictReader(io.StringIO(await get_text('../../examples/recordlink/patient-registry-a.csv'))))
source_b = list(csv.DictReader(io.StringIO(await get_text('../../examples/recordlink/surveillance-b.csv'))))
truth = list(csv.DictReader(io.StringIO(await get_text('../../examples/recordlink/true-links.csv'))))
expected = json.loads(await get_text('../../examples/recordlink/expected-recordlink-results.json'))
assert len(source_a) == len(source_b) == 8 and len(truth) == 5
{'source_a': len(source_a), 'source_b': len(source_b), 'truth_links': len(truth)}

In [ ]:
def normalize(value):
    text = unicodedata.normalize('NFKC', str(value)).strip().casefold()
    return text or None
def normalize_fuzzy(value):
    text = normalize(value)
    return re.sub(r'[^\w]+', ' ', text, flags=re.UNICODE).strip() if text else None
def jaro_winkler(left, right):
    if left == right: return 1.0
    if not left or not right: return 0.0
    distance = max(0, max(len(left), len(right)) // 2 - 1)
    lm, rm = [False] * len(left), [False] * len(right); matches = 0
    for i, char in enumerate(left):
        for j in range(max(0, i-distance), min(i+distance+1, len(right))):
            if rm[j] or char != right[j]: continue
            lm[i] = rm[j] = True; matches += 1; break
    if not matches: return 0.0
    ml = [c for i, c in enumerate(left) if lm[i]]; mr = [c for i, c in enumerate(right) if rm[i]]
    transpositions = sum(a != b for a, b in zip(ml, mr)) / 2
    jaro = (matches/len(left) + matches/len(right) + (matches-transpositions)/matches) / 3
    prefix = 0
    while prefix < min(4, len(left), len(right)) and left[prefix] == right[prefix]: prefix += 1
    return jaro + prefix * 0.1 * (1-jaro) if jaro > 0.7 else jaro
assert abs(jaro_winkler('martha', 'marhta') - 0.9611111111111111) < 1e-15

In [ ]:
exact_pairs = [('date_of_birth','DOB'), ('sex','SEX'), ('art_code','ART_CODE')]
fuzzy_pairs = [('first_name','given_name'), ('last_name','family_name'), ('patient_address','Address')]
threshold = expected['comparison']['fuzzyThreshold']
candidates = [(a, b) for a in source_a for b in source_b if normalize(a['facility_code']) == normalize(b['site_code'])]
scores = []
for a, b in candidates:
    exact = sum(normalize(a[x]) is not None and normalize(a[x]) == normalize(b[y]) for x, y in exact_pairs)
    similarities = [jaro_winkler(normalize_fuzzy(a[x]), normalize_fuzzy(b[y])) for x, y in fuzzy_pairs]
    scores.append(exact + sum(value >= threshold for value in similarities))
distribution = [{'score': score, 'candidates': count} for score, count in sorted(Counter(scores).items(), reverse=True)]
classes = ['match' if score >= expected['classification']['matchThreshold'] else 'review' if score >= expected['classification']['reviewThreshold'] else 'non-match' for score in scores]
class_counts = dict(Counter(classes))
truth_pairs = {(row['source_a_id'], row['source_b_id']) for row in truth}
proposed_matches = {(a['record_id'], b['client_id']) for (a, b), label in zip(candidates, classes) if label == 'match'}
tp = len(proposed_matches & truth_pairs); fp = len(proposed_matches - truth_pairs); fn = len(truth_pairs - proposed_matches)
precision = tp / (tp + fp) if tp + fp else 0; recall = tp / len(truth_pairs) if truth_pairs else 1
metrics = {'truePositiveMatches': tp, 'falsePositiveMatches': fp, 'falseNegativeMatches': fn, 'truthLinksInReview': sum(label == 'review' and (a['record_id'], b['client_id']) in truth_pairs for (a, b), label in zip(candidates, classes)), 'precision': precision, 'recall': recall, 'f1': 2*precision*recall/(precision+recall) if precision+recall else 0}
assert len(candidates) == expected['sourceCounts']['blockedCandidatePairs'] == 7
assert scores == expected['comparison']['candidateScoresInBlockingOrder']
assert distribution == expected['comparison']['scoreDistribution']
assert classes == expected['classification']['classesInBlockingOrder']
assert class_counts == expected['classification']['counts']
assert metrics == expected['classification']['truthMetrics']
assert {(a['record_id'], b['client_id']) for a, b in candidates} >= truth_pairs
{'scores': scores, 'classes': classes, 'metrics': metrics, 'status': 'PASS'}

In [ ]:
import hashlib
program = await get_text('../../examples/recordlink/recordlink-command-tour.pgm7')
canonical_plan = next(line for line in program.splitlines() if line.startswith('EPIAI RECORDLINK '))
candidate_indexes = [(source_a.index(a), source_b.index(b)) for a, b in candidates]
binding = {'plan': canonical_plan, 'pairs': candidate_indexes, 'scores': scores, 'classes': classes}
fingerprint = hashlib.sha256(json.dumps(binding, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
artifact = {'kind': 'epi-info-ai.recordlink-review', 'schemaVersion': 1, 'candidateSet': {'candidates': len(candidates), 'fingerprint': fingerprint}, 'decisions': [{'candidateNumber': 5, 'decision': 'match', 'reason': 'acceptable-variation'}], 'privacy': {'identifiersIncluded': False, 'recordValuesIncluded': False, 'normalizedValuesIncluded': False}}
serialized = json.dumps(artifact, sort_keys=True)
assert len(fingerprint) == 64 and artifact['candidateSet']['candidates'] == 7
assert all(value not in serialized for value in ['A004', 'B004', '1988-12-01', '1988-12-02'])
tampered = dict(binding); tampered['scores'] = scores[:-1] + [1]
assert hashlib.sha256(json.dumps(tampered, sort_keys=True, separators=(',', ':')).encode()).hexdigest() != fingerprint
{'fingerprint': fingerprint, 'decisions': len(artifact['decisions']), 'privacy': artifact['privacy'], 'status': 'PASS'}

In [ ]:
def cluster_proposal(edges):
    parent = {('A', i): ('A', i) for i in range(len(source_a))} | {('B', i): ('B', i) for i in range(len(source_b))}
    members = {node: {node[0]} for node in parent}
    def root(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]; node = parent[node]
        return node
    accepted, rejected = 0, []
    for score, ordinal, a_index, b_index in sorted(edges, key=lambda edge: (-edge[0], edge[1])):
        left, right = root(('A', a_index)), root(('B', b_index))
        if left == right: continue
        if members[left] & members[right]: rejected.append(ordinal); continue
        keep, remove = sorted([left, right])
        parent[remove] = keep; members[keep] |= members.pop(remove); accepted += 1
    roots = {root(node) for node in parent}
    sizes = [sum(root(node) == cluster for node in parent) for cluster in roots]
    return {'acceptedEdges': accepted, 'rejectedConflicts': len(rejected), 'linkedClusters': sum(size > 1 for size in sizes), 'singletons': sum(size == 1 for size in sizes), 'totalPeople': len(sizes)}, rejected
accepted_ordinals = [i for i, label in enumerate(classes, 1) if label == 'match'] + [5]
edges = [(scores[i-1], i, *candidate_indexes[i-1]) for i in accepted_ordinals]
cluster_result, rejected = cluster_proposal(edges)
assert cluster_result == expected['personClustering']['reviewCandidate5AsMatch']
conflict_edges = edges + [(scores[0], 8, candidate_indexes[0][0], candidate_indexes[3][1])]
conflict_result, rejected = cluster_proposal(conflict_edges)
assert rejected == [8] and conflict_result['rejectedConflicts'] == 1
{'reviewed_match': cluster_result, 'source-membership_conflict_candidates': rejected, 'status': 'PASS'}

## Result and boundary

A clean run independently reproduces the browser candidate order, score distribution, threshold classes, truth-set precision/recall/F1, aggregate-only review binding, the 11-person V0.7 proposal, and deliberate conflict rejection. Review decisions remain explicit; the lab does not write audit tables, merge, or modify records.